## Training notebook

In [1]:
import gym
from gym.wrappers import FrameStack, FlattenObservation
import highway_env

from stable_baselines3 import PPO
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact
import ipywidgets as widgets
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.env_checker import check_env
from highway_env.tb_callback import TensorboardCallback
from torch.multiprocessing import set_start_method
try:
     set_start_method('spawn')
except RuntimeError:
    pass


from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO','WINDOWS_FORNO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'WINDOWS_FORNO', 'UBUNTU_DELL…

<function __main__.f(desired_os)>

In [3]:
if os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/elios/pighetti/HighwayDRL/training_output'
elif os_id.value == 'UBUNTU_PIGO':
    OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output'
elif os_id.value == 'WINDOWS_FORNO':
    OUTPUT_PATH = r'C:/Users/luka-/OneDrive/Documenti/PhD/HighwayDRL/training_output'

### Select environment

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-multi-agent-v0','dm-env-v0','multipleO-dm-env-v0','highway-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-multi-agent-v0', 'dm-env-v0', 'multipleO-…

<function __main__.f(input_env)>

In [5]:
total_timesteps = 3e5
num_envs = 5
# Create and wrap the environment
env = make_vec_env(env_id.value, n_envs=num_envs) # , num_envs=num_envs
# env = VecNormalize(env, clip_obs=1, norm_obs_keys=['kinematics'])
# env = FlattenObservation(env)
# env = VecFrameStack(env, n_stack=4)

# check_env(env)
# env.configure({
#     "training_total_timesteps": total_timesteps
# })

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=1000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='adv_')

# eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
#                              log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
#                              deterministic=True, render=False)

tb_callback = TensorboardCallback()

In [6]:
ppo_ilr = 3.5e-4
# PPO parameters
model = PPO("MultiInputPolicy",
        env,
        gamma=0.997,
        batch_size=128,
        n_steps=4096,
        n_epochs=10,
        ent_coef=0.1,
        verbose=0,
        learning_rate=ppo_ilr,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

In [7]:
try:
    # Train the agent for n steps
    model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, tb_callback], progress_bar=True)
finally:
    # Save the trained agent
    model.save(f'{OUTPUT_PATH}/models/adv_train_newRF.zip')

Output()

## Continued learning

In [4]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)
env_cl = Monitor(env_cl, filename='../../training_output/log_cl') 
check_env(env_cl)

/home/pigo/miniconda3/envs/highway-env/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [5]:
model_cl = PPO.load(f"{OUTPUT_PATH}/final_models/ppo_standard_200k_FORNO", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}/training_output/logdir")

In [6]:
# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/training_output/checkpoints',
                                         name_prefix='dm_rl_callback_CL')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

tb_callback = TensorboardCallback()

In [ ]:
new_timesteps = 2e5
try:
    model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, tb_callback], progress_bar=True)
finally:
    model_cl.save(f'{OUTPUT_PATH}/training_output/models/ppo_FORNO_conservative_curriculum_200k_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')